In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph

In [2]:
def ByOBJPathFast(path: str,
                         transposeAxes: bool = True,
                         mantissa: int = None,
                         silent: bool = False):
    """
    Imports OBJ faces as fast as possible and returns a flat list of Topologic faces.

    This method intentionally ignores:
    - groups / objects
    - materials
    - texture coordinates
    - normals
    - lines
    - points
    - dictionaries
    - clusters
    - self-merging
    - coplanar merging
    - topology hierarchy construction

    Supported OBJ primitives
    ------------------------
    v : vertices
    f : faces

    Parameters
    ----------
    path : str
        The path to the OBJ file.

    transposeAxes : bool , optional
        If True, converts OBJ coordinates from (x, y, z) to (x, -z, y).
        Default is True.

    mantissa : int , optional
        If set to an integer, vertex coordinates are rounded to this number of
        decimal places. If None, no rounding is performed. Default is None.

    silent : bool , optional
        If True, suppresses error messages. Default is False.

    Returns
    -------
    list
        A flat list of Topologic face objects.
    """
    from topologicpy.Topology import Topology

    if not isinstance(path, str) or len(path) == 0:
        if not silent:
            print("ByOBJPathFast - Error: The input objPath is not a valid string. Returning an empty list.")
        return []
    
    
    with open(path, 'r') as obj_file:
        obj_string = obj_file.read()

    verts = []
    faces_idx = []

    do_round = isinstance(mantissa, int)

    def _resolve_index(i, n):
        # OBJ indices are 1-based. Negative indices are relative to the current vertex list.
        return i - 1 if i > 0 else n + i

    # -------------------------------------------------------------------------
    # Fast OBJ parsing: only v and f
    # -------------------------------------------------------------------------
    for raw in obj_string.splitlines():
        if not raw:
            continue

        c0 = raw[0]

        # Vertex line: v x y z
        if c0 == "v" and len(raw) > 1 and raw[1].isspace():
            parts = raw.split()
            if len(parts) < 4:
                continue

            try:
                x = float(parts[1])
                y = float(parts[2])
                z = float(parts[3])
            except Exception:
                continue

            if do_round:
                x = round(x, mantissa)
                y = round(y, mantissa)
                z = round(z, mantissa)

            if transposeAxes:
                verts.append([x, -z, y])
            else:
                verts.append([x, y, z])

        # Face line: f v1 v2 v3 ...
        elif c0 == "f" and len(raw) > 1 and raw[1].isspace():
            parts = raw.split()
            if len(parts) < 4:
                continue

            n_verts = len(verts)
            idxs = []
            ok = True

            for token in parts[1:]:
                # Supports:
                # f 1 2 3
                # f 1/1 2/2 3/3
                # f 1//1 2//2 3//3
                # f 1/1/1 2/2/2 3/3/3
                slash = token.find("/")
                if slash >= 0:
                    token = token[:slash]

                if not token:
                    ok = False
                    break

                try:
                    idx = _resolve_index(int(token), n_verts)
                except Exception:
                    ok = False
                    break

                if idx < 0 or idx >= n_verts:
                    ok = False
                    break

                idxs.append(idx)

            if ok and len(idxs) >= 3:
                faces_idx.append(idxs)

    # -------------------------------------------------------------------------
    # Build one Topologic face per OBJ face.
    # Use local face vertices to avoid passing the entire OBJ vertex table each time.
    # -------------------------------------------------------------------------
    faces = []

    for idxs in faces_idx:
        try:
            local_vertices = [verts[i] for i in idxs]
            local_face = list(range(len(local_vertices)))

            face = Topology.ByGeometry(
                vertices=local_vertices,
                edges=[],
                faces=[local_face]
            )

            if face is not None:
                faces.append(face)
        except Exception:
            continue

    return faces

In [3]:
objPath = r"c:\Users\sarwj\Downloads\building_new.obj"
faces = ByOBJPathFast(objPath)
print("Imported", len(faces), "faces.")

Imported 8874 faces.


In [ ]:
Topology.Show(faces)

In [ ]:
cc = CellComplex.ByFaces(faces)
print(cc)

In [ ]:
g = Graph.ByTopology(cc)
print(g)

In [ ]:
verts = Graph.Vertices(g)
for v in verts:
    d = Dictionary.ByKeysValues(["size", "color"], [5, "yellow"])
    v = Topology.SetDictionary(v, d)
Topology.Show(g, vertexSizeKey="size", vertexColorKey="color")